# Quick start

This guide demonstrates a minimal working example of the `qiskit-addon-obp` package. We use operator backpropagation (OBP) to reduce the depth of a quantum circuit by absorbing trailing gates into the observable. To see examples of how to build realistic workflows with this tool and execute on quantum hardware, check out the tutorials on the IBM Quantum Platform ([OBP Tutorial](https://quantum.cloud.ibm.com/docs/en/tutorials/operator-back-propagation)).

## Prepare inputs for OBP

OBP takes as input a list of circuit slices and an observable. It backpropagates slices from the end of the circuit into the observable, reducing circuit depth at the cost of additional Pauli terms.

We generate a time-evolution circuit for a 10-qubit Heisenberg model and slice it by gate type.

In [1]:
import numpy as np
from qiskit.transpiler import CouplingMap
from qiskit.quantum_info import SparsePauliOp
from qiskit.synthesis import LieTrotter
from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian, generate_time_evolution_circuit
from qiskit_addon_utils.slicing import slice_by_gate_types

# Generate a circuit to reduce
coupling_map = CouplingMap.from_heavy_hex(3, bidirectional=False)
reduced_coupling_map = coupling_map.reduce([0, 13, 1, 14, 10, 16, 5, 12, 8, 18])

hamiltonian = generate_xyz_hamiltonian(
    reduced_coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)

# Slice the circuit and define an observable
slices = slice_by_gate_types(circuit)
observable = SparsePauliOp("IIIIIIIIIZ")

print(f"Original circuit depth: {circuit.depth()}")
print(f"Number of slices: {len(slices)}")
print(f"Observable terms: {len(observable)}")

Original circuit depth: 18
Number of slices: 18
Observable terms: 1


## Reduce circuit depth with OBP

We call `backpropagate` to absorb slices into the observable. The function returns the expanded observable, the remaining (shorter) circuit slices, and metadata about the procedure.

In [2]:
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_utils.slicing import combine_slices

bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=OperatorBudget(max_qwc_groups=8)
)

reduced_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} of {len(slices)} slices.")
print(f"Reduced circuit depth: {reduced_circuit.depth()}")
print(f"Expanded observable terms: {len(bp_obs)}")

Backpropagated 7 of 18 slices.
Reduced circuit depth: 11
Expanded observable terms: 18
